In [4]:
#EXERCÍCIO 1: Regressão Logística (Chatbot Bancário)

# =====================================================================
# LAB 1: NLU BANCÁRIO COM REGRESSÃO LOGÍSTICA
# =====================================================================

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

dados_treino = [
    ("quero consultar meu saldo", "consultar_saldo"),
    ("qual é o meu saldo", "consultar_saldo"),
    ("quanto tenho na conta", "consultar_saldo"),
    ("ver saldo da conta", "consultar_saldo"),

    ("quero fazer um pix", "fazer_pix"),
    ("como faço um pix", "fazer_pix"),
    ("preciso enviar um pix", "fazer_pix"),
    ("quero transferir dinheiro via pix", "fazer_pix"),

    ("quero pagar uma conta", "pagar_conta"),
    ("preciso pagar um boleto", "pagar_conta"),
    ("como pago minha conta", "pagar_conta"),
    ("quero pagar o boleto", "pagar_conta"),

    ("quero bloquear meu cartão", "bloquear_cartao"),
    ("bloqueie meu cartão", "bloquear_cartao"),
    ("preciso bloquear o cartão", "bloquear_cartao"),
    ("meu cartão foi perdido", "bloquear_cartao"),

    ("quero sacar dinheiro", "sacar_dinheiro"),
    ("preciso fazer um saque", "sacar_dinheiro"),
    ("como faço um saque", "sacar_dinheiro"),
    ("quero retirar dinheiro", "sacar_dinheiro"),

    ("quero depositar dinheiro", "depositar_dinheiro"),
    ("preciso fazer um depósito", "depositar_dinheiro"),
    ("como deposito dinheiro", "depositar_dinheiro"),
    ("quero colocar dinheiro na conta", "depositar_dinheiro"),

    ("quero falar com um atendente", "fora_escopo"),
    ("preciso de ajuda", "fora_escopo"),
    ("quero falar com uma pessoa", "fora_escopo"),
    ("não sei o que fazer", "fora_escopo"),

    ("quero fazer um pix de 100 reais", "fazer_pix"),
    ("quero fazer um pix de 150 reais", "fazer_pix"),
    ("faça um pix de 200 reais", "fazer_pix"),
    ("enviar 150 reais por pix", "fazer_pix"),
    ("quero enviar 100 reais via pix", "fazer_pix"),
    ("preciso mandar 150 reais por pix", "fazer_pix"),

]

stopwords = {
    "a", "as", "o", "os", "um", "uma",
    "de", "do", "da", "dos", "das",
    "e", "em", "no", "na", "nos", "nas",
    "meu", "minha", "meus", "minhas",
    "é", "ser", "para", "por", "com",
    "que", "qual"
}


# 1. PRÉ-PROCESSAMENTO
def pré_processar(texto):
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower()).strip()
    tokens = texto_limpo.split()

    # TODO 1.1: Filtre as stopwords da lista 'tokens' usando List Comprehension
    tokens_filtrados = [
        token for token in tokens if token not in stopwords
    ]

    return " ".join(tokens_filtrados)


X_treino_limpo = [pré_processar(item[0]) for item in dados_treino]
y_treino = [item[1] for item in dados_treino]


# 2. VETORIZAÇÃO E TREINAMENTO
vectorizer = TfidfVectorizer()

# TODO 1.2: Aprenda o vocabulário e vetorize 'X_treino_limpo' em um único passo
X_vetorizado = vectorizer.fit_transform(X_treino_limpo)

modelo = LogisticRegression()
modelo.fit(X_vetorizado, y_treino)


# 3. EXTRAÇÃO DE ENTIDADE (REGEX)
def extrair_valor_reais(texto_original):
    match = re.search(r'(\d+)\s*reais', texto_original, re.IGNORECASE)

    # TODO 1.3: Se houver match, converta o grupo 1 para float.
    # Caso contrário, retorne None.
    return float(match.group(1)) if match else None


# 4. PIPELINE NLU
def processar_nlu(mensagem_usuario, threshold=0.55):
    texto_p = pré_processar(mensagem_usuario)
    vetor_input = vectorizer.transform([texto_p])

    probas = modelo.predict_proba(vetor_input)[0]

    # TODO 1.4:
    # Extraia a MAIOR probabilidade (max)
    # e a intenção prevista (argmax em modelo.classes_)
    maior_confianca = max(probas)
    intencao_prevista = modelo.classes_[probas.argmax()]

    # TODO 1.5:
    # Se maior_confianca < threshold ou intencao == "fora_escopo",
    # retorne FALLBACK.
    # Caso contrário, retorne SUCESSO com a intenção,
    # confiança e o valor extraído.

    if maior_confianca < threshold or intencao_prevista == "fora_escopo":
        return {
            "status": "FALLBACK",
            "mensagem": "Não consegui identificar sua solicitação."
        }
    else:
        return {
            "status": "SUCESSO",
            "intencao": intencao_prevista,
            "confianca": maior_confianca,
            "valor": extrair_valor_reais(mensagem_usuario)
        }


# --- TESTE 1 ---
if __name__ == "__main__":
    print(processar_nlu("quero fazer um pix de 150 reais"))


{'status': 'SUCESSO', 'intencao': np.str_('fazer_pix'), 'confianca': np.float64(0.655410157265827), 'valor': 150.0}


In [6]:
#EXERCÍCIO 2: Naive Bayes (Filtro de Spam / Triagem)

# =====================================================================
# LAB 2: TRIAGEM DE CHAMADOS COM NAIVE BAYES
# =====================================================================

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

dados_treino = [
    ("Urgente: o servidor caiu e a aplicacao esta fora", "incidente_critico"),
    ("Erro grave no banco de dados e sistema parou", "incidente_critico"),
    ("Servidor caiu e sistema esta fora do ar", "incidente_critico"),
    ("Sistema caiu", "incidente_critico"),
    ("Servidor fora do ar", "incidente_critico"),
    ("Erro no servidor urgente", "incidente_critico"),
    ("Banco de dados parou", "incidente_critico"),

    ("Como faco para alterar minha senha do e-mail", "duvida_suporte"),
    ("Qual o procedimento para solicitar novo mouse", "duvida_suporte"),
    ("Preciso trocar minha senha", "duvida_suporte"),
    ("Como altero minha senha", "duvida_suporte"),

    ("Gostaria de saber o horario de funcionamento", "fora_escopo"),
    ("Qual o horario de atendimento", "fora_escopo"),

]

stopwords = ["o", "a", "e", "do", "da", "para", "de"]

# 1. PRÉ-PROCESSAMENTO
def pré_processar(texto):
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower()).strip()
    tokens = texto_limpo.split()

    # TODO 2.1: Filtre as stopwords mantendo os tokens válidos
    tokens_filtrados = [
        token for token in tokens if token not in stopwords
    ]

    return " ".join(tokens_filtrados)

X_treino_limpo = [pré_processar(item[0]) for item in dados_treino]
y_treino = [item[1] for item in dados_treino]

# 2. VETORIZAÇÃO E TREINAMENTO (NAIVE BAYES)
vectorizer = TfidfVectorizer()
X_vetorizado = vectorizer.fit_transform(X_treino_limpo)

# TODO 2.2: Instancie o modelo Naive Bayes (MultinomialNB)
modelo_nb = MultinomialNB()

# TODO 2.3: Treine o modelo com X_vetorizado e y_treino
modelo_nb.fit(X_vetorizado, y_treino)

# 3. EXTRAÇÃO DE ENTIDADE (CÓDIGO DE PROTOCOLO)
def extrair_protocolo(texto):
    # Procura um padrão de protocolo como 'INC-1234' ou 'INC1234'
    match = re.search(r'INC-?\d{4}', texto, re.IGNORECASE)

    # TODO 2.4: Retorne a string do protocolo encontrado
    # em maiúsculas ou None
    return match.group(0).upper() if match else None

# 4. PIPELINE NLU
def processar_triagem(mensagem_usuario, threshold=0.60):
    texto_p = pré_processar(mensagem_usuario)
    vetor_input = vectorizer.transform([texto_p])

    probas = modelo_nb.predict_proba(vetor_input)[0]

    maior_confianca = max(probas)
    intencao_prevista = modelo_nb.classes_[probas.argmax()]

    # TODO 2.5
    if maior_confianca < threshold:
        return {
            "status": "FALLBACK",
            "mensagem": "Não consegui classificar o chamado."
        }
    else:
        return {
            "status": "SUCESSO",
            "intencao": str(intencao_prevista),
            "confianca": float(maior_confianca),
            "protocolo": extrair_protocolo(mensagem_usuario)
        }

# --- TESTE 2 ---
if __name__ == "__main__":
    print(processar_triagem(
        "Urgente: servidor caiu no protocolo INC-9982"
    ))


{'status': 'SUCESSO', 'intencao': 'incidente_critico', 'confianca': 0.7967002273124675, 'protocolo': 'INC-9982'}


In [8]:
#EXERCÍCIO 3: Árvore de Decisão (Chatbot de E-Commerce)

# =====================================================================
# LAB 3: CHATBOT DE E-COMMERCE COM DECISION TREE
# =====================================================================

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier

dados_treino = [
    ("Onde esta meu pedido e qual o rastreio", "rastrear_pedido"),
    ("Quero saber a localizacao da minha encomenda", "rastrear_pedido"),
    ("Quero cancelar minha compra e pedir reembolso", "cancelar_compra"),
    ("Preciso cancelar o pedido feito ontem", "cancelar_compra"),
    ("Qual o horario de atendimento da loja fisica", "fora_escopo")
]

stopwords = ["o", "a", "meu", "minha", "e", "da", "do"]

# 1. PRÉ-PROCESSAMENTO
def pré_processar(texto):
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower()).strip()
    tokens = texto_limpo.split()

    # TODO 3.1: Filtre as stopwords da frase
    tokens_filtrados = [
        token for token in tokens if token not in stopwords
    ]

    return " ".join(tokens_filtrados)

X_treino_limpo = [pré_processar(item[0]) for item in dados_treino]
y_treino = [item[1] for item in dados_treino]

# 2. VETORIZAÇÃO E TREINAMENTO (DECISION TREE)
vectorizer = TfidfVectorizer()
X_vetorizado = vectorizer.fit_transform(X_treino_limpo)

# TODO 3.2: Instancie o modelo DecisionTreeClassifier
modelo_tree = DecisionTreeClassifier()

# TODO 3.3: Treine o modelo com os dados vetorizados e os rótulos de treino
modelo_tree.fit(X_vetorizado, y_treino)

# 3. EXTRAÇÃO DE ENTIDADE (CÓDIGO DE RASTREIO BR)
def extrair_codigo_rastreio(texto):
    # Procura códigos de rastreio como 'BR123456789'
    match = re.search(r'BR\d{9}', texto, re.IGNORECASE)

    # TODO 3.4: Se encontrar, retorne em maiúsculas. Se não, None.
    return match.group(0).upper() if match else None

# 4. PIPELINE NLU
def processar_ecommerce(mensagem_usuario, threshold=0.50):
    texto_p = pré_processar(mensagem_usuario)
    vetor_input = vectorizer.transform([texto_p])

    # DecisionTree também calcula probabilidades
    # com base na pureza das folhas
    probas = modelo_tree.predict_proba(vetor_input)[0]

    maior_confianca = max(probas)
    intencao_prevista = modelo_tree.classes_[probas.argmax()]

    # TODO 3.5: Verificação de segurança
    if maior_confianca < threshold or intencao_prevista == "fora_escopo":
        return {
            "status": "FALLBACK",
            "mensagem": "Não consegui identificar sua solicitação."
        }
    else:
        return {
            "status": "SUCESSO",
            "intencao": str(intencao_prevista),
            "confianca": float(maior_confianca),
            "codigo_rastreio": extrair_codigo_rastreio(mensagem_usuario)
        }

# --- TESTE 3 ---
if __name__ == "__main__":
    print(processar_ecommerce(
        "Quero rastrear o pedido BR987654321"
    ))


{'status': 'SUCESSO', 'intencao': 'rastrear_pedido', 'confianca': 1.0, 'codigo_rastreio': 'BR987654321'}
